# APAR generation — Ahmednagar (sorghum (jowar))

Computes APAR = MODIS fAPAR (MOD15A2H) × PAR (ERA5-Land radiation × 0.48) for each 8-day period, masks it to the crop and exports it per grid cell.

**Inputs:** grid shapefile (`shape/*_pt25*.shp`), Earth Engine crop-mask asset  
**Outputs:** `apar/APAR-<YYYY_MM_DD>-<n>.tif` (×1000, int16)  
**Next:** `02_raster_formation/raster_formation_ahmednagar`  

> Update the path variables in the first cells before running. See [`docs/`](../../docs/) for methodology and parameters.

In [ ]:
import geemap
import ee
import os
import numpy as np

In [ ]:
from datetime import datetime

In [ ]:
ee.Authenticate()
ee.Initialize()

In [ ]:
base=r'C:\Local\Desktop_previous\miscellaneous\Neha\YieldData\21Jan\ahmadnagar\apar'

In [ ]:
shapeFile=r'C:\Local\Desktop_previous\miscellaneous\Neha\YieldData\21Jan\ahmadnagar\shape\ahmadnagar_pt25_selected.shp'
gp_data=geemap.shp_to_ee(shapeFile)
boundary = geemap.shp_to_ee(shapeFile)
cropmask = ee.Image('projects/ee-gaurpushkar8/assets/Ahmednagar_Rabi2022-23_Jowar_gcs')

In [ ]:
roi = boundary.first()
geom = roi.geometry()

In [ ]:
gp_bound=gp_data.getInfo()['features']

In [ ]:
# /// Growth stages


g0 = '2022-11-01'
g1 = '2022-12-01'
g2 = '2022-12-31'
g3 = '2023-30-01'
g4 = '2023-02-28'


In [ ]:
def typeCast(image):
    return image.toFloat()

In [ ]:
# // Temperature 

tmax = 33
tmin = 5
topt = 27

RUE = 2.8
HI = 0.6


In [ ]:
def temperaturestress(image):
    numerator = (image.subtract(ee.Image(tmin))).multiply(image.subtract(ee.Image(tmax))) 
    denominator = ((image.subtract(ee.Image(tmin))).multiply(image.subtract(ee.Image(tmax)))).subtract(
                  (image.subtract(ee.Image(topt))).multiply((image.subtract(ee.Image(topt)))))
    return typeCast(numerator.divide(denominator).rename('Tstress'))
  



In [ ]:
def local_max(ndwi):
    
    theMax = ndwi.reduceRegion({'reducer' : ee.Reducer.max(), 
                                'geometry' : boundary.geometry(), 
                                'maxPixels' : 74882817, 
                                'bestEffort' : True}
                               )
    return ndwi.set({'NDWI':theMax})

In [ ]:
# // water stress calculation

def waterscale(image):
    img = image.clip(boundary).multiply(0.0001)
    nir = image.select('sur_refl_b02')
    swir = image.select('sur_refl_b06')
    ndwi = nir.subtract(swir).divide(nir.add(swir)).rename('NDWI')
    gcs = (cropmask.projection())
    ndwi = ndwi.reproject(gcs)
    ndwi = ndwi.mask(cropmask)
#     theMax = ndwi.reduceRegion({'reducer' : ee.Reducer.max(), 
#                                 'geometry' : boundary.geometry(), 
# #                                 'maxPixels' : 74882817, 
#                                 'bestEffort' : True}
#                                )
#     theMax = local_max(ndwi)
    ws = ee.Image(1).subtract(ndwi).divide(ee.Image(1.37))
    ws = ws.reproject(gcs)
    return typeCast(ws)



In [ ]:
def fparscale(image):
    return typeCast(image).clip(boundary).multiply(0.01).mask(cropmask);

In [ ]:
def insolscale(image):
    return typeCast(image).clip(boundary).multiply(0.48).divide(1000000).mask(cropmask);

In [ ]:
def image_download_yield(image,filename,bound):
    image=image.multiply(1000)
    image = image.clip(bound)
    image = image.int16()
    if not os.path.exists(filename):
        print(filename)
        geemap.ee_export_image(image, filename=filename, scale=10, region=bound, file_per_band=False)

In [ ]:
def image_download(image,filename,bound):
    geemap.ee_export_image(image, filename=filename, scale=10, region=bound, file_per_band=False)

In [ ]:
def gp_image_download(image,date,factor):
    for d in np.arange(len(gp_bound)):
        fet = gp_bound[d]
        feature=ee.Feature(gp_data.filterMetadata('Id','equals',fet['properties']['Id']).first())
        fn=os.path.join(base,factor+'-'+date+'-'+str(d)+'.tif')
        image_download_yield(image,fn,feature.geometry())

In [ ]:
# // APAR calculation 
 
mod15a2h = ee.ImageCollection("MODIS/061/MOD15A2H").select('Fpar_500m').filter(ee.Filter.date(g0,g4)).filterBounds(boundary);
fpar = mod15a2h.map(fparscale);

apar_imgs = ee.List([]);

def parprocessing(date_i,date_e,date):
    global apar_imgs
    era5land = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").select('surface_solar_radiation_downwards_sum').filter(ee.Filter.date(date_i,date_e)).filterBounds(boundary);
    par = era5land.map(insolscale);
    par_sum = par.sum().set('system:index',date);
    gcs = cropmask.projection();
    par_sum = par_sum.rename([date]);
    fpar_img = fpar.filterMetadata('system:index','equals',date).first().reproject(gcs);
    apar_img = typeCast(par_sum).multiply(typeCast(fpar_img)).rename([date]).set('system:index',date);
    apar_img = apar_img.clip(boundary)
    apar_img = apar_img.mask(cropmask);
    fn=os.path.join(base,'apar-',date+'.tif')
    gp_image_download(apar_img,date,'APAR')
    apar_imgs = apar_imgs.add(typeCast(apar_img));


fparinfo = (fpar.getInfo()['features'])
for d in np.arange(len(fparinfo)):
    date_i='-'.join(fparinfo[d]['properties']['system:index'].split('_'))
    print(date_i)
    if(d==len(fparinfo)-1):
        date_e=g4

    else:
        date_e='-'.join(fparinfo[d+1]['properties']['system:index'].split('_'))


    parprocessing(date_i,date_e,fparinfo[d]['properties']['system:index'])


apar = ee.ImageCollection(apar_imgs)


In [ ]:

# mod09a1 = ee.ImageCollection('MODIS/061/MOD09A1').filter(ee.Filter.date(g2, g3)).filterBounds(boundary);
# ndwi_coll = mod09a1.mean()
# waterScaler = waterscale(ndwi_coll);
# waterScaler = waterScaler.clip(boundary) 
# fn=os.path.join(base,'apar','waterscaler.tif')
# gp_image_download(waterScaler,'2','WS')


# era5_temp = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").select('temperature_2m').filter(ee.Filter.date(g2, g3)).filterBounds(boundary);
# era5_mean_temp = era5_temp.mean().subtract(ee.Image(273.15));
# tstress = temperaturestress(era5_mean_temp);
# tstress = tstress.clip(boundary).mask(cropmask)
# fn=os.path.join(base,'apar','temp_stress.tif')
# gp_image_download(tstress,'2','TS')
